# MalthusJAX vs Evosax Benchmark (Updated API)

This notebook compares MalthusJAX with Evosax using the current engine API:
- Uses `engine.init_state()` + `engine.step()` / `engine._step_legacy()` loop
- Compatible with the Shell & Kernel architecture
- Handles compilation scenarios: not compiled, compiled (not warmed), compiled (warmed)
- Tests BBOB benchmark functions with statistical analysis

## Important Notes

**FORCE_LEGACY Mode**: Set to `True` by default during operator migration. The FAST_LANE execution path requires all operators to support the batched `apply_kernel` interface. Once operators are fully migrated, set `FORCE_LEGACY = False` to enable FAST_LANE mode.

**Fitness Evaluation**: BBOB problems are minimization tasks (`maximize=False`). Lower fitness values are better. The benchmark compares solution quality (best_cost), runtime, and throughput across frameworks and compilation scenarios.

**Random Seed Handling**: Each run uses `BASE_SEED + run_index` to ensure reproducibility. Warmup runs use a different seed offset to avoid contaminating measured results.

In [17]:
# Imports
import jax
import jax.numpy as jnp
import jax.random as jr
import malthusjax as mjx
from malthusjax.core.fitness.bbob_evaluator import BBOBEvaluator, BBOBConfig
from malthusjax.engine.genetic_engine import GeneticEngine, GeneticEngineParams
from malthusjax.operators.selection.elite_pool import ElitePoolSelection
from malthusjax.operators.crossover.real import UniformCrossover
from malthusjax.operators.mutation.real import GaussianMutation
from evosax.algorithms.population_based.simple_ga import SimpleGA
from evosax.problems import BBOBProblem
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print(f"JAX version: {jax.__version__}")
print(f"JAX devices: {jax.devices()}")

JAX version: 0.8.0
JAX devices: [CpuDevice(id=0)]


## 🔄 Reload Modules

The engine code was fixed but Python may have cached the old version. Let's force a reload.

In [18]:
# Force reload of the malthusjax module to get the fixed engine code
import importlib
import sys

# Remove cached modules
modules_to_reload = [k for k in sys.modules.keys() if k.startswith('malthusjax')]
for module in modules_to_reload:
    del sys.modules[module]

# Reimport
import malthusjax as mjx
from malthusjax.core.fitness.bbob_evaluator import BBOBEvaluator, BBOBConfig
from malthusjax.engine.genetic_engine import GeneticEngine, GeneticEngineParams
from malthusjax.operators.selection.elite_pool import ElitePoolSelection
from malthusjax.operators.crossover.real import UniformCrossover
from malthusjax.operators.mutation.real import GaussianMutation

print("✅ MalthusJAX modules reloaded with fixed optimization direction!")

✅ MalthusJAX modules reloaded with fixed optimization direction!


## Configuration Parameters

All benchmark parameters in one place for easy modification.

In [19]:
# ============================================================================
# BENCHMARK CONFIGURATION - MODIFY THESE PARAMETERS
# ============================================================================

# Problem Configuration
PROBLEM_NAME = "sphere"            # BBOB function name (sphere, rastrigin, rosenbrock, etc.)
DIMENSIONS = 10                    # Problem dimensionality
BOUNDS = (-5.0, 5.0)               # Search space bounds
FIXED_PROBLEM_SEED = 0             # Fixed landscape seed (same problem for all runs)

# Evolution Parameters
POPULATION_SIZE = 100              # Population size
NUM_GENERATIONS = 50               # Number of generations
CROSSOVER_RATE = 0.5               # Crossover probability
ELITE_RATIO = 0.2                  # Proportion of elites to select
MUTATION_RATE = 0.3                # Mutation probability
MUTATION_STRENGTH = 0.1            # Gaussian mutation strength

# Statistical Parameters
NUM_RUNS = 5                       # Number of independent runs for statistics
BASE_SEED = 42                     # Base random seed (incremented for each run)

# Scenario Configuration
RUN_NOT_COMPILED = True            # Run scenario: not compiled
RUN_COMPILED_NOT_WARMED = True     # Run scenario: compiled but not warmed (includes compile time)
RUN_COMPILED_WARMED = True         # Run scenario: compiled and warmed (steady state)

# Engine Mode (NEW: control FAST_LANE vs LEGACY)
FORCE_LEGACY = False               # ✅ FAST_LANE now supported! UniformCrossover & GaussianMutation
                                   # use jax.vmap() to handle batched keys

# Display
VERBOSE = True                     # Print progress during runs

# Derived parameters
ELITE_POOL_SIZE = int(POPULATION_SIZE * ELITE_RATIO)

print("📋 Benchmark Configuration")
print("=" * 70)
print(f"Problem:          {PROBLEM_NAME} ({DIMENSIONS}D)")
print(f"Population Size:  {POPULATION_SIZE:,}")
print(f"Generations:      {NUM_GENERATIONS}")
print(f"Independent Runs: {NUM_RUNS}")
print(f"Base Seed:        {BASE_SEED}")
print(f"Force Legacy:     {FORCE_LEGACY}")
print(f"\nScenarios to run:")
print(f"  Not Compiled:           {RUN_NOT_COMPILED}")
print(f"  Compiled (Not Warmed):  {RUN_COMPILED_NOT_WARMED}")
print(f"  Compiled (Warmed):      {RUN_COMPILED_WARMED}")
print("=" * 70)

📋 Benchmark Configuration
Problem:          sphere (10D)
Population Size:  100
Generations:      50
Independent Runs: 5
Base Seed:        42
Force Legacy:     False

Scenarios to run:
  Not Compiled:           True
  Compiled (Not Warmed):  True
  Compiled (Warmed):      True


## Helper Functions

Setup builders using the current engine API.

In [20]:
# ============================================================================
# HELPER FUNCTIONS (CURRENT ENGINE API)
# ============================================================================

def setup_bbob_problem():
    """Setup BBOB problem and evaluator with fixed seed."""
    # BBOB problems are minimization tasks
    bbob_config = BBOBConfig(
        fn_name=PROBLEM_NAME,
        num_dims=DIMENSIONS,
        seed=FIXED_PROBLEM_SEED,
        maximize=False  # Minimization
    )
    mjx_evaluator = BBOBEvaluator.create(bbob_config)
    
    # Evosax problem (for comparison)
    evosax_problem = BBOBProblem(PROBLEM_NAME, num_dims=DIMENSIONS, seed=FIXED_PROBLEM_SEED)
    
    return mjx_evaluator, evosax_problem


def build_malthusjax_engine(evaluator):
    """Build MalthusJAX engine using current API with concrete operator classes."""
    genome_config = mjx.RealGenomeConfig(length=DIMENSIONS, bounds=BOUNDS)
    
    # Use GeneticEngineParams from the engine module
    params = GeneticEngineParams(pop_size=POPULATION_SIZE, elitism=0)
    
    # Concrete operator instances
    selection = ElitePoolSelection(
        num_selections=POPULATION_SIZE,
        elite_k=ELITE_POOL_SIZE
    )
    
    crossover = UniformCrossover(
        num_offspring=1,
        crossover_rate=CROSSOVER_RATE
    )
    
    mutation = GaussianMutation(
        num_offspring=1,
        mutation_rate=MUTATION_RATE,
        mutation_strength=MUTATION_STRENGTH
    )
    
    engine = GeneticEngine(
        genome_config=genome_config,
        evaluator=evaluator,
        selection=selection,
        crossover=crossover,
        mutation=mutation,
    )
    
    return engine, params


def build_evosax_strategy(problem):
    """Build Evosax strategy with configured parameters."""
    rng = jax.random.PRNGKey(0)
    init_solution = problem.sample(rng)
    
    strategy = SimpleGA(
        population_size=POPULATION_SIZE,
        solution=init_solution
    )
    
    es_params = strategy.default_params.replace(crossover_rate=CROSSOVER_RATE)
    
    return strategy, es_params


print("✅ Helper functions loaded")

✅ Helper functions loaded


## Single Experiment Runners

Functions to run a single experiment for each framework with compilation and warmup toggles.

In [21]:
# ============================================================================
# MALTHUSJAX SINGLE EXPERIMENT RUNNER (CURRENT API)
# ============================================================================

def run_malthusjax_single(
    seed: int,
    compile: bool,
    warmup: bool,
    verbose: bool = False
) -> Dict[str, float]:
    """Run a single MalthusJAX experiment using init_state + step loop."""
    
    # Setup
    mjx_evaluator, _ = setup_bbob_problem()
    engine, params = build_malthusjax_engine(mjx_evaluator)
    
    # Initialize state with the provided seed
    key = jr.PRNGKey(seed)
    
    # Initialize state for the measured run
    state = engine.init_state(key, params)
    next_key = state.rng_key
    
    # Measured run: iterate generations using step API
    if compile and not FORCE_LEGACY:
        # JIT-compiled evolution loop using lax.scan for performance
        # Removed try/except to avoid nested loop in HLO compilation
        def scan_step(carry, _):
            key, state = carry
            new_key, new_state, metrics = engine.step(key, state, params)
            return (new_key, new_state), None
        
        def evolution_loop(key, state):
            return jax.lax.scan(scan_step, (key, state), None, length=NUM_GENERATIONS)
        
        # JIT the entire evolution loop
        jit_evolution = jax.jit(evolution_loop)
        
        # Warmup if requested (run the actual compiled code to warm JIT cache)
        if warmup:
            warmup_key = jr.PRNGKey(seed + 10000)
            warmup_state = engine.init_state(warmup_key, params)
            (_, warmup_final_state), _ = jit_evolution(warmup_state.rng_key, warmup_state)
            jax.block_until_ready(warmup_final_state.best_fitness)
        
        # Measured run
        start_time = time.time()
        (next_key, state), _ = jit_evolution(next_key, state)
        jax.block_until_ready(state.best_fitness)
        runtime = time.time() - start_time
    else:
        # Python loop for non-compiled or FORCE_LEGACY mode
        # Warmup if requested
        if warmup:
            warmup_key = jr.PRNGKey(seed + 10000)
            warmup_state = engine.init_state(warmup_key, params)
            warmup_next_key = warmup_state.rng_key
            for _ in range(3):
                if FORCE_LEGACY:
                    warmup_next_key, warmup_state, _ = engine._step_legacy(warmup_next_key, warmup_state, params)
                else:
                    warmup_next_key, warmup_state, _ = engine.step(warmup_next_key, warmup_state, params)
            jax.block_until_ready(warmup_state.best_fitness)
        
        # Measured run
        start_time = time.time()
        for _ in range(NUM_GENERATIONS):
            if FORCE_LEGACY:
                next_key, state, metrics = engine._step_legacy(next_key, state, params)
            else:
                next_key, state, metrics = engine.step(next_key, state, params)
        jax.block_until_ready(state.best_fitness)
        runtime = time.time() - start_time
    
    # Best cost: for BBOB minimization with maximize=False, report raw fitness
    best_cost = float(state.best_fitness)
    throughput = (POPULATION_SIZE * NUM_GENERATIONS) / runtime if runtime > 0 else float('inf')
    
    if verbose:
        print(f"  Seed {seed}: Cost={best_cost:.6f}, Time={runtime:.4f}s, "
              f"Throughput={throughput/1e6:.2f}M evals/s")
    
    return {
        'seed': seed,
        'best_cost': best_cost,
        'runtime': runtime,
        'throughput': throughput,
        'framework': 'MalthusJAX',
        'compile': compile,
        'warmup': warmup
    }


print("✅ MalthusJAX runner loaded")

✅ MalthusJAX runner loaded


In [22]:
# ============================================================================
# EVOSAX SINGLE EXPERIMENT RUNNER
# ============================================================================

def run_evosax_single(
    seed: int,
    compile: bool,
    warmup: bool,
    verbose: bool = False
) -> Dict[str, float]:
    """Run a single Evosax experiment with compilation/warmup control."""
    
    # Setup
    _, problem = setup_bbob_problem()
    strategy, es_params = build_evosax_strategy(problem)
    
    # Define step function
    def step_impl(state, param_state, rng):
        rng_ask, rng_eval, rng_tell = jax.random.split(rng, 3)
        x, state = strategy.ask(rng_ask, state, es_params)
        fitness, new_param_state, _ = problem.eval(rng_eval, x, param_state)
        state, metrics = strategy.tell(rng_tell, x, fitness, state, es_params)
        return state, new_param_state, fitness
    
    step = jax.jit(step_impl) if compile else step_impl
    
    # Define evolution loop
    def scan_step(carry, _):
        state, prob_state, rng = carry
        rng, rng_step = jax.random.split(rng)
        new_state, new_prob_state, _ = step(state, prob_state, rng_step)
        return (new_state, new_prob_state, rng), None
    
    def run_evolution_impl(state, prob_state, rng):
        return jax.lax.scan(
            scan_step,
            (state, prob_state, rng),
            None,
            length=NUM_GENERATIONS
        )
    
    run_evolution = jax.jit(run_evolution_impl) if compile else run_evolution_impl
    
    # Initialize
    rng = jr.PRNGKey(seed)
    rng, rng_init, rng_pop = jax.random.split(rng, 3)
    
    initial_pop = jax.random.uniform(
        rng_pop,
        (POPULATION_SIZE, DIMENSIONS),
        minval=BOUNDS[0],
        maxval=BOUNDS[1]
    )
    initial_fitness = jnp.full((POPULATION_SIZE,), jnp.inf)
    state = strategy.init(rng_init, initial_pop, initial_fitness, es_params)
    prob_state = problem.init(jr.PRNGKey(FIXED_PROBLEM_SEED))
    
    # Warmup if requested
    if warmup:
        (_state, _prob_state, _rng), _ = run_evolution(state, prob_state, rng)
        jax.block_until_ready(_state.best_fitness)
    
    # Measured run
    start_time = time.time()
    (state, prob_state, rng), _ = run_evolution(state, prob_state, rng)
    jax.block_until_ready(state.best_fitness)
    runtime = time.time() - start_time
    
    # Extract metrics
    best_cost = float(state.best_fitness)
    throughput = (POPULATION_SIZE * NUM_GENERATIONS) / runtime
    
    if verbose:
        print(f"  Seed {seed}: Cost={best_cost:.6f}, Time={runtime:.4f}s, "
              f"Throughput={throughput/1e6:.2f}M evals/s")
    
    return {
        'seed': seed,
        'best_cost': best_cost,
        'runtime': runtime,
        'throughput': throughput,
        'framework': 'Evosax',
        'compile': compile,
        'warmup': warmup
    }


print("✅ Evosax runner loaded")

✅ Evosax runner loaded


## Benchmark Orchestrator

Run experiments across all scenarios and frameworks.

In [23]:
# ============================================================================
# BENCHMARK ORCHESTRATOR
# ============================================================================

def run_benchmark_for_scenarios(
    num_runs: int,
    verbose: bool = True
) -> pd.DataFrame:
    """Run benchmark across all enabled scenarios."""
    
    scenarios = []
    
    if RUN_NOT_COMPILED:
        scenarios.append({
            'name': 'not_compiled',
            'label': 'Not Compiled',
            'compile': False,
            'warmup': False
        })
    
    if RUN_COMPILED_NOT_WARMED:
        scenarios.append({
            'name': 'compiled_not_warmed',
            'label': 'Compiled (Not Warmed)',
            'compile': True,
            'warmup': False
        })
    
    if RUN_COMPILED_WARMED:
        scenarios.append({
            'name': 'compiled_warmed',
            'label': 'Compiled (Warmed)',
            'compile': True,
            'warmup': True
        })
    
    all_results = []
    
    print("\n🚀 Running Benchmark Across All Scenarios")
    print("=" * 70)
    
    for scenario in scenarios:
        if verbose:
            print(f"\n📊 Scenario: {scenario['label']}")
            print(f"   Compile={scenario['compile']}, Warmup={scenario['warmup']}")
            print("-" * 70)
        
        # Run MalthusJAX
        if verbose:
            print("\n  🔧 MalthusJAX:")
        
        for i in range(num_runs):
            seed = BASE_SEED + i
            result = run_malthusjax_single(
                seed=seed,
                compile=scenario['compile'],
                warmup=scenario['warmup'],
                verbose=verbose
            )
            result['scenario'] = scenario['name']
            result['scenario_label'] = scenario['label']
            all_results.append(result)
        
        # Run Evosax
        if verbose:
            print("\n  🥊 Evosax:")
        
        for i in range(num_runs):
            seed = BASE_SEED + i
            result = run_evosax_single(
                seed=seed,
                compile=scenario['compile'],
                warmup=scenario['warmup'],
                verbose=verbose
            )
            result['scenario'] = scenario['name']
            result['scenario_label'] = scenario['label']
            all_results.append(result)
    
    df = pd.DataFrame(all_results)
    
    if verbose:
        print("\n" + "=" * 70)
        print("✅ Benchmark complete!")
        print("=" * 70)
    
    return df


print("✅ Orchestrator loaded")

✅ Orchestrator loaded


## Statistical Analysis Functions

In [24]:
# ============================================================================
# STATISTICAL ANALYSIS
# ============================================================================

def compute_statistics(df: pd.DataFrame, metric: str) -> Dict[str, float]:
    """Compute statistical summary for a metric."""
    values = df[metric].values
    
    mean_val = np.mean(values)
    std_val = np.std(values, ddof=1)
    median_val = np.median(values)
    min_val = np.min(values)
    max_val = np.max(values)
    
    n = len(values)
    if n > 1:
        sem = std_val / np.sqrt(n)
        ci = stats.t.interval(0.95, n-1, loc=mean_val, scale=sem)
    else:
        ci = (mean_val, mean_val)
    
    return {
        'mean': mean_val,
        'std': std_val,
        'median': median_val,
        'min': min_val,
        'max': max_val,
        'ci_lower': ci[0],
        'ci_upper': ci[1],
        'n': n
    }


def compare_frameworks_by_scenario(df_all: pd.DataFrame) -> pd.DataFrame:
    """Statistical comparison between frameworks grouped by scenario."""
    
    results = []
    
    for scenario in df_all['scenario_label'].unique():
        df_scenario = df_all[df_all['scenario_label'] == scenario]
        
        for framework in ['MalthusJAX', 'Evosax']:
            df_fw = df_scenario[df_scenario['framework'] == framework]
            
            cost_stats = compute_statistics(df_fw, 'best_cost')
            time_stats = compute_statistics(df_fw, 'runtime')
            throughput_stats = compute_statistics(df_fw, 'throughput')
            
            results.append({
                'Scenario': scenario,
                'Framework': framework,
                'Cost (mean±std)': f"{cost_stats['mean']:.6f} ± {cost_stats['std']:.6f}",
                'Runtime (mean±std)': f"{time_stats['mean']:.4f} ± {time_stats['std']:.4f}",
                'Throughput (M/s)': f"{throughput_stats['mean']/1e6:.2f} ± {throughput_stats['std']/1e6:.2f}",
            })
    
    comparison_df = pd.DataFrame(results)
    
    # Statistical tests per scenario
    print("\n" + "="*80)
    print("STATISTICAL TESTS BY SCENARIO")
    print("="*80)
    
    for scenario in df_all['scenario_label'].unique():
        df_scenario = df_all[df_all['scenario_label'] == scenario]
        
        mjx_cost = df_scenario[df_scenario['framework'] == 'MalthusJAX']['best_cost'].values
        evosax_cost = df_scenario[df_scenario['framework'] == 'Evosax']['best_cost'].values
        mjx_time = df_scenario[df_scenario['framework'] == 'MalthusJAX']['runtime'].values
        evosax_time = df_scenario[df_scenario['framework'] == 'Evosax']['runtime'].values
        
        if len(mjx_cost) > 1 and len(evosax_cost) > 1:
            cost_ttest = stats.ttest_ind(mjx_cost, evosax_cost, equal_var=False)
            time_ttest = stats.ttest_ind(mjx_time, evosax_time, equal_var=False)
            
            print(f"\n📊 {scenario}")
            print(f"   Cost: t={cost_ttest.statistic:.4f}, p={cost_ttest.pvalue:.4e}")
            print(f"   Time: t={time_ttest.statistic:.4f}, p={time_ttest.pvalue:.4e}")
            
            if cost_ttest.pvalue < 0.05:
                winner = "MalthusJAX" if np.mean(mjx_cost) < np.mean(evosax_cost) else "Evosax"
                print(f"   ✓ {winner} has significantly better solution quality")
            
            if time_ttest.pvalue < 0.05:
                winner = "MalthusJAX" if np.mean(mjx_time) < np.mean(evosax_time) else "Evosax"
                print(f"   ✓ {winner} is significantly faster")
    
    print("="*80 + "\n")
    
    return comparison_df


print("✅ Statistical analysis functions loaded")

✅ Statistical analysis functions loaded


## Run Benchmark

Execute the full benchmark across all scenarios.

## Diagnostic: Fitness Evaluation Check

Quick sanity check to verify fitness evaluation is working correctly.

In [25]:
# Diagnostic: Check fitness evaluation
print("🔍 Diagnostic: Testing fitness evaluation...")

# Create a simple test genome at origin (should have fitness ~0 for sphere)
mjx_evaluator, evosax_problem = setup_bbob_problem()

# Create proper genome objects (RealGenome only needs 'values' field)
from malthusjax.core.genome.real_genome import RealGenome

# Test genomes at two known points
test_genome_zero = RealGenome(values=jnp.zeros(DIMENSIONS))
test_genome_ones = RealGenome(values=jnp.ones(DIMENSIONS))

print(f"\nMalthusJAX BBOBEvaluator:")
print(f"  Config: maximize={mjx_evaluator.config.maximize}")
fitness_zero = mjx_evaluator.evaluate(test_genome_zero)
fitness_ones = mjx_evaluator.evaluate(test_genome_ones)
print(f"  Fitness at origin (zeros): {fitness_zero:.6f}")
print(f"  Fitness at ones: {fitness_ones:.6f}")

# Test Evosax evaluation
print(f"\nEvosax BBOBProblem:")
test_key = jr.PRNGKey(0)
evosax_state = evosax_problem.init(test_key)
test_pop = jnp.stack([jnp.zeros(DIMENSIONS), jnp.ones(DIMENSIONS)])
fitness_evosax, _, _ = evosax_problem.eval(test_key, test_pop, evosax_state)
print(f"  Fitness at origin (zeros): {fitness_evosax[0]:.6f}")
print(f"  Fitness at ones: {fitness_evosax[1]:.6f}")

print("\n✅ Expected for Sphere function (minimization):")
print("   - Fitness at origin should be ~0")
print("   - Fitness at ones should be ~10 (sum of 10 * 1^2)")
print("\n🔍 If MalthusJAX shows large positive values, there may be a sign issue")
print("🔍 Check if the evaluator is applying maximize=False correctly")

🔍 Diagnostic: Testing fitness evaluation...

MalthusJAX BBOBEvaluator:
  Config: maximize=False
  Fitness at origin (zeros): 103.747650
  Fitness at ones: 110.300140

Evosax BBOBProblem:
  Fitness at origin (zeros): 103.747650
  Fitness at ones: 110.300140

✅ Expected for Sphere function (minimization):
   - Fitness at origin should be ~0
   - Fitness at ones should be ~10 (sum of 10 * 1^2)

🔍 If MalthusJAX shows large positive values, there may be a sign issue
🔍 Check if the evaluator is applying maximize=False correctly

MalthusJAX BBOBEvaluator:
  Config: maximize=False
  Fitness at origin (zeros): 103.747650
  Fitness at ones: 110.300140

Evosax BBOBProblem:
  Fitness at origin (zeros): 103.747650
  Fitness at ones: 110.300140

✅ Expected for Sphere function (minimization):
   - Fitness at origin should be ~0
   - Fitness at ones should be ~10 (sum of 10 * 1^2)

🔍 If MalthusJAX shows large positive values, there may be a sign issue
🔍 Check if the evaluator is applying maximize=Fals

## 🐛 BUG FOUND: RNG Key Not Being Used

**Root Cause**: The benchmark loop ignores the returned RNG key from `engine.step()` / `engine._step_legacy()`.

The engine API returns `(next_key, new_state, metrics)`, but the benchmark loop uses:
```python
_, state, metrics = engine._step_legacy(state.rng_key, state, params)
```

This means `state.rng_key` is always the same value across all generations! The evolution is deterministic because it reuses the same RNG key every iteration.

**Fix**: Use the returned key for the next iteration.

## 🔬 Verification: Test RNG Key Progression

Let's verify that the fix is working by running a mini-test showing RNG key changes across generations.

In [26]:
# Quick test: verify RNG key changes across steps
print("🔬 Testing RNG key progression...")
mjx_evaluator, _ = setup_bbob_problem()
engine, params = build_malthusjax_engine(mjx_evaluator)

test_seed = 999
key = jr.PRNGKey(test_seed)
state = engine.init_state(key, params)

print(f"\nInitial state.rng_key: {state.rng_key}")
print(f"Initial best_fitness: {state.best_fitness:.6f}")

# Track key across 3 steps
next_key = state.rng_key
for i in range(3):
    print(f"\nGeneration {i+1}:")
    print(f"  Input key: {next_key}")
    next_key, state, metrics = engine._step_legacy(next_key, state, params)
    print(f"  Output key: {next_key}")
    print(f"  Best fitness: {state.best_fitness:.6f}")

print("\n✅ If keys change each iteration, the fix is working!")
print("❌ If keys stay the same, the function definition wasn't reloaded!")

🔬 Testing RNG key progression...

Initial state.rng_key: [2891654438 1268651290]
Initial best_fitness: 98.483215

Generation 1:
  Input key: [2891654438 1268651290]
  Output key: [1155558511  451012745]
  Best fitness: 98.483215

Generation 2:
  Input key: [1155558511  451012745]
  Output key: [3212576575  442014973]
  Best fitness: 98.483215

Generation 3:
  Input key: [3212576575  442014973]
  Output key: [1500797183  680293291]
  Best fitness: 98.483215

✅ If keys change each iteration, the fix is working!
❌ If keys stay the same, the function definition wasn't reloaded!

Initial state.rng_key: [2891654438 1268651290]
Initial best_fitness: 98.483215

Generation 1:
  Input key: [2891654438 1268651290]
  Output key: [1155558511  451012745]
  Best fitness: 98.483215

Generation 2:
  Input key: [1155558511  451012745]
  Output key: [3212576575  442014973]
  Best fitness: 98.483215

Generation 3:
  Input key: [3212576575  442014973]
  Output key: [1500797183  680293291]
  Best fitness: 9

## 🚨 CRITICAL BUG DISCOVERED: Engine Optimization Direction

**The RNG fix works correctly** (keys change each iteration ✅), BUT there's a more fundamental bug in the engine:

**Bug Location**: `GeneticEngine._update_hall_of_fame()` (line 258-266 in `genetic_engine.py`)

**Problem**: The method uses hardcoded maximization logic:
```python
best_idx = jnp.argmax(new_pop.fitness)  # Always finds maximum
is_new_record = curr_best_fit > state.best_fitness  # Always checks if greater
new_best_fit = jnp.maximum(state.best_fitness, curr_best_fit)  # Always takes max
```

This means the engine **always maximizes**, even when `evaluator.config.maximize=False`!

**Evidence from verification test**: Fitness increases (98 → 334 → 386 → 505) even though we're supposed to minimize.

**Why identical fitness across runs**: The engine converges to the same (wrong) maximum value regardless of seed because the optimization direction is flipped.

**Fix Required**: The engine needs to respect `self.evaluator.config.maximize` in `_update_hall_of_fame()`, similar to how `init_state()` handles it with `opt_sign`.

## ✅ FAST_LANE Mode Now Supported!

**Status**: `FORCE_LEGACY = False` now works for this benchmark.

**Fixed Operators**:
- ✅ **`UniformCrossover`** - Updated `apply_kernel()` to use `jax.vmap()` for batched bernoulli calls
- ✅ **`GaussianMutation`** - Updated `apply_kernel()` to use `jax.vmap()` for batched normal calls

**How it Works**:
```python
# When FAST_LANE is enabled, operators receive batched keys (pop_size, 2)
if keys.ndim == 2:  # Batched keys
    # Use vmap to apply random function to each key independently
    def make_mask_single(key):
        return jar.bernoulli(key, p=rate, shape=single_shape)
    mask = jax.vmap(make_mask_single)(keys)
else:  # Single key (legacy path)
    mask = jar.bernoulli(keys, p=rate, shape=data.shape)
```

**Performance Benefits**: With FAST_LANE enabled, MalthusJAX uses fully JIT-compiled `jax.lax.scan()` loops instead of Python loops, providing **10-100x speedup** in the warmed scenario!

In [27]:
# Run the full benchmark
df_all_results = run_benchmark_for_scenarios(NUM_RUNS, verbose=VERBOSE)


🚀 Running Benchmark Across All Scenarios

📊 Scenario: Not Compiled
   Compile=False, Warmup=False
----------------------------------------------------------------------

  🔧 MalthusJAX:
  Seed 42: Cost=94.336319, Time=16.3084s, Throughput=0.00M evals/s
  Seed 42: Cost=94.336319, Time=16.3084s, Throughput=0.00M evals/s
  Seed 43: Cost=78.078018, Time=15.2323s, Throughput=0.00M evals/s
  Seed 43: Cost=78.078018, Time=15.2323s, Throughput=0.00M evals/s
  Seed 44: Cost=76.089470, Time=22.2638s, Throughput=0.00M evals/s
  Seed 44: Cost=76.089470, Time=22.2638s, Throughput=0.00M evals/s
  Seed 45: Cost=77.828751, Time=15.5960s, Throughput=0.00M evals/s
  Seed 45: Cost=77.828751, Time=15.5960s, Throughput=0.00M evals/s
  Seed 46: Cost=97.581490, Time=16.8435s, Throughput=0.00M evals/s

  🥊 Evosax:
  Seed 46: Cost=97.581490, Time=16.8435s, Throughput=0.00M evals/s

  🥊 Evosax:
  Seed 42: Cost=46.449093, Time=0.3653s, Throughput=0.01M evals/s
  Seed 42: Cost=46.449093, Time=0.3653s, Throughput

## Statistical Analysis

In [28]:
# Perform statistical comparison
comparison_df = compare_frameworks_by_scenario(df_all_results)

print("\n" + "="*80)
print("STATISTICAL SUMMARY BY SCENARIO")
print("="*80)
display(comparison_df)


STATISTICAL TESTS BY SCENARIO

📊 Not Compiled
   Cost: t=7.9587, p=1.2430e-03
   Time: t=13.1644, p=1.9217e-04
   ✓ Evosax has significantly better solution quality
   ✓ Evosax is significantly faster

📊 Compiled (Not Warmed)
   Cost: t=7.9587, p=1.2430e-03
   Time: t=-1.1227, p=2.9507e-01
   ✓ Evosax has significantly better solution quality

📊 Compiled (Warmed)
   Cost: t=7.9587, p=1.2430e-03
   Time: t=-0.2608, p=8.0705e-01
   ✓ Evosax has significantly better solution quality


STATISTICAL SUMMARY BY SCENARIO


,Scenario,Framework,Cost (mean±std),Runtime (mean±std),Throughput (M/s)
0,Not Compiled,MalthusJAX,84.782809 ± 10.295182,17.2488 ± 2.8722,0.00 ± 0.00
1,Not Compiled,Evosax,47.958080 ± 1.027221,0.3390 ± 0.0230,0.01 ± 0.00
2,Compiled (Not Warmed),MalthusJAX,84.782809 ± 10.295182,0.3348 ± 0.0183,0.01 ± 0.00
3,Compiled (Not Warmed),Evosax,47.958080 ± 1.027221,0.3468 ± 0.0153,0.01 ± 0.00
4,Compiled (Warmed),MalthusJAX,84.782809 ± 10.295182,0.0033 ± 0.0004,1.54 ± 0.16
5,Compiled (Warmed),Evosax,47.958080 ± 1.027221,0.0033 ± 0.0000,1.51 ± 0.01


## Raw Data & Export

In [29]:
# View all results
print("\n📊 All Experimental Results:")
display(df_all_results)

# Save to CSV
csv_filename = f"benchmark_results_{PROBLEM_NAME}_{DIMENSIONS}D_new_api.csv"
df_all_results.to_csv(csv_filename, index=False)
print(f"\n💾 Results saved to '{csv_filename}'")


📊 All Experimental Results:


,seed,best_cost,runtime,throughput,framework,compile,warmup,scenario,scenario_label
0,42,94.336319,16.308397,3.065905e+02,MalthusJAX,False,False,not_compiled,Not Compiled
1,43,78.078018,15.232348,3.282488e+02,MalthusJAX,False,False,not_compiled,Not Compiled
2,44,76.089470,22.263829,2.245795e+02,MalthusJAX,False,False,not_compiled,Not Compiled
3,45,77.828751,15.595999,3.205950e+02,MalthusJAX,False,False,not_compiled,Not Compiled
4,46,97.581490,16.843452,2.968513e+02,MalthusJAX,False,False,not_compiled,Not Compiled
5,42,46.449093,0.365308,1.368707e+04,Evosax,False,False,not_compiled,Not Compiled
6,43,48.525261,0.357736,1.397678e+04,Evosax,False,False,not_compiled,Not Compiled
7,44,47.339302,0.310087,1.612451e+04,Evosax,False,False,not_compiled,Not Compiled
8,45,48.841820,0.338594,1.476696e+04,Evosax,False,False,not_compiled,Not Compiled
9,46,48.634922,0.323399,1.546077e+04,Evosax,False,False,not_compiled,Not Compiled



💾 Results saved to 'benchmark_results_sphere_10D_new_api.csv'


## Summary Report

In [30]:
# Generate final summary
print("=" * 80)
print("FINAL SUMMARY")
print("=" * 80)

print(f"\n📈 Problem: {PROBLEM_NAME.upper()} ({DIMENSIONS}D)")
print(f"   Population: {POPULATION_SIZE:,} | Generations: {NUM_GENERATIONS} | Runs: {NUM_RUNS}")
print(f"   Force Legacy: {FORCE_LEGACY}")

# Track initial and final fitness for each framework
mjx_initial_fitness = {}
mjx_final_fitness = {}
evosax_initial_fitness = {}
evosax_final_fitness = {}

for scenario_label in df_all_results['scenario_label'].unique():
    print(f"\n📊 {scenario_label}:")
    df_scenario = df_all_results[df_all_results['scenario_label'] == scenario_label]
    
    for framework in ['MalthusJAX', 'Evosax']:
        df_fw = df_scenario[df_scenario['framework'] == framework]
        mean_cost = df_fw['best_cost'].mean()
        mean_time = df_fw['runtime'].mean()
        mean_throughput = df_fw['throughput'].mean() / 1e6
        
        print(f"   {framework}:")
        print(f"      Cost:       {mean_cost:.6f}")
        print(f"      Runtime:    {mean_time:.4f}s")
        print(f"      Throughput: {mean_throughput:.2f} M evals/s")
        
        # Track for improvement calculation
        if scenario_label == 'Not Compiled':
            if framework == 'MalthusJAX':
                mjx_initial_fitness[scenario_label] = mean_cost
            else:
                evosax_initial_fitness[scenario_label] = mean_cost

# Calculate and display fitness improvements
print("\n" + "=" * 80)
print("📈 AVERAGE FITNESS IMPROVEMENT (Lower is Better for Minimization)")
print("=" * 80)

# Get initial fitness from first generation (we need to track this differently)
# For now, show the improvement from Not Compiled to Compiled scenarios
df_not_compiled = df_all_results[df_all_results['scenario_label'] == 'Not Compiled']
df_compiled_warmed = df_all_results[df_all_results['scenario_label'] == 'Compiled (Warmed)']

for framework in ['MalthusJAX', 'Evosax']:
    initial_cost = df_not_compiled[df_not_compiled['framework'] == framework]['best_cost'].mean()
    final_cost = df_compiled_warmed[df_compiled_warmed['framework'] == framework]['best_cost'].mean()
    improvement = ((initial_cost - final_cost) / initial_cost) * 100
    
    print(f"\n{framework}:")
    print(f"   Initial Cost (Not Compiled):     {initial_cost:.6f}")
    print(f"   Final Cost (Compiled Warmed):    {final_cost:.6f}")
    print(f"   Improvement:                     {improvement:.2f}%")
    print(f"   Absolute Reduction:              {initial_cost - final_cost:.6f}")

# Compare frameworks
mjx_final = df_compiled_warmed[df_compiled_warmed['framework'] == 'MalthusJAX']['best_cost'].mean()
evosax_final = df_compiled_warmed[df_compiled_warmed['framework'] == 'Evosax']['best_cost'].mean()
fitness_gap = abs(mjx_final - evosax_final)
relative_gap = (fitness_gap / evosax_final) * 100

print("\n" + "-" * 80)
print("Framework Comparison (Compiled Warmed):")
print(f"   MalthusJAX:  {mjx_final:.6f}")
print(f"   Evosax:      {evosax_final:.6f}")
print(f"   Gap:         {fitness_gap:.6f} ({relative_gap:.2f}%)")

print(f"\n📊 Files Generated:")
print(f"   ✓ {csv_filename}")

print("\n" + "=" * 80)
print("✅ Benchmark Complete!")
print("=" * 80)

FINAL SUMMARY

📈 Problem: SPHERE (10D)
   Population: 100 | Generations: 50 | Runs: 5
   Force Legacy: False

📊 Not Compiled:
   MalthusJAX:
      Cost:       84.782809
      Runtime:    17.2488s
      Throughput: 0.00 M evals/s
   Evosax:
      Cost:       47.958080
      Runtime:    0.3390s
      Throughput: 0.01 M evals/s

📊 Compiled (Not Warmed):
   MalthusJAX:
      Cost:       84.782809
      Runtime:    0.3348s
      Throughput: 0.01 M evals/s
   Evosax:
      Cost:       47.958080
      Runtime:    0.3468s
      Throughput: 0.01 M evals/s

📊 Compiled (Warmed):
   MalthusJAX:
      Cost:       84.782809
      Runtime:    0.0033s
      Throughput: 1.54 M evals/s
   Evosax:
      Cost:       47.958080
      Runtime:    0.0033s
      Throughput: 1.51 M evals/s

📈 AVERAGE FITNESS IMPROVEMENT (Lower is Better for Minimization)

MalthusJAX:
   Initial Cost (Not Compiled):     84.782809
   Final Cost (Compiled Warmed):    84.782809
   Improvement:                     0.00%
   Absolute 

## 🔍 Compilation Analysis: View Lowered HLO

Let's examine what JAX is actually compiling to understand the performance difference.

In [31]:
# ============================================================================
# COMPILATION ANALYSIS: View Lowered HLO for MalthusJAX
# ============================================================================

print("🔍 Analyzing MalthusJAX Compilation...")
print("=" * 80)

# Setup
mjx_evaluator, _ = setup_bbob_problem()
engine, params = build_malthusjax_engine(mjx_evaluator)

# Initialize state
test_key = jr.PRNGKey(42)
state = engine.init_state(test_key, params)

# Define the scan-based evolution loop (same as in benchmark - no try/except)
def scan_step(carry, _):
    key, state = carry
    new_key, new_state, metrics = engine.step(key, state, params)
    return (new_key, new_state), None

def evolution_loop(key, state):
    return jax.lax.scan(scan_step, (key, state), None, length=NUM_GENERATIONS)

# Get lowered HLO
print("\n📊 Compiling evolution_loop...")
jit_evolution = jax.jit(evolution_loop)

# Trigger compilation by running once
(next_key, final_state), _ = jit_evolution(test_key, state)
jax.block_until_ready(final_state.best_fitness)

# Get the lowered representation
print("\n🔬 Getting lowered HLO representation...")
lowered = jax.jit(evolution_loop).lower(test_key, state)

print("\n" + "=" * 80)
print("LOWERED HLO (first 5000 chars):")
print("=" * 80)
hlo_text = lowered.as_text()
print(hlo_text[:5000])
print("\n... (truncated)")
print(f"\nTotal HLO length: {len(hlo_text):,} characters")

# Save full HLO to file
hlo_filename = f"malthusjax_hlo_{PROBLEM_NAME}_{DIMENSIONS}D_clean.txt"
with open(hlo_filename, 'w') as f:
    f.write(hlo_text)
print(f"\n💾 Full HLO saved to: {hlo_filename}")

print("\n" + "=" * 80)
print("✅ Compilation analysis complete!")
print("=" * 80)

🔍 Analyzing MalthusJAX Compilation...



📊 Compiling evolution_loop...

🔬 Getting lowered HLO representation...

LOWERED HLO (first 5000 chars):
module @jit_evolution_loop attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<2xui32>, %arg1: tensor<100x10xf32>, %arg2: tensor<100xf32>, %arg3: tensor<10xf32>, %arg4: tensor<i32>, %arg5: tensor<f32>, %arg6: tensor<i32>, %arg7: tensor<2xui32>, %arg8: tensor<100xf32>) -> (tensor<2xui32> {jax.result_info = "result[0][0]"}, tensor<100x10xf32> {jax.result_info = "result[0][1].population.genes.values"}, tensor<100xf32> {jax.result_info = "result[0][1].population.fitness"}, tensor<10xf32> {jax.result_info = "result[0][1].best_genome.values"}, tensor<i32> {jax.result_info = "result[0][1].generation"}, tensor<f32> {jax.result_info = "result[0][1].best_fitness"}, tensor<i32> {jax.result_info = "result[0][1].stagnation_counter"}, tensor<2xui32> {jax.result_info = "result[0][1].rng_key"}, tensor<100xf32> {jax.result_info = "result[0

## 🔍 Compilation Analysis: View Lowered HLO for Evosax

Compare with Evosax to see the difference in compiled structure.

In [32]:
# ============================================================================
# COMPILATION ANALYSIS: View Lowered HLO for Evosax
# ============================================================================

print("🔍 Analyzing Evosax Compilation...")
print("=" * 80)

# Setup
_, problem = setup_bbob_problem()
strategy, es_params = build_evosax_strategy(problem)

# Define step function
def step_impl(state, param_state, rng):
    rng_ask, rng_eval, rng_tell = jax.random.split(rng, 3)
    x, state = strategy.ask(rng_ask, state, es_params)
    fitness, new_param_state, _ = problem.eval(rng_eval, x, param_state)
    state, metrics = strategy.tell(rng_tell, x, fitness, state, es_params)
    return state, new_param_state, fitness

step = jax.jit(step_impl)

# Define evolution loop
def scan_step(carry, _):
    state, prob_state, rng = carry
    rng, rng_step = jax.random.split(rng)
    new_state, new_prob_state, _ = step(state, prob_state, rng_step)
    return (new_state, new_prob_state, rng), None

def run_evolution_impl(state, prob_state, rng):
    return jax.lax.scan(
        scan_step,
        (state, prob_state, rng),
        None,
        length=NUM_GENERATIONS
    )

run_evolution = jax.jit(run_evolution_impl)

# Initialize
rng = jr.PRNGKey(42)
rng, rng_init, rng_pop = jax.random.split(rng, 3)

initial_pop = jax.random.uniform(
    rng_pop,
    (POPULATION_SIZE, DIMENSIONS),
    minval=BOUNDS[0],
    maxval=BOUNDS[1]
)
initial_fitness = jnp.full((POPULATION_SIZE,), jnp.inf)
state = strategy.init(rng_init, initial_pop, initial_fitness, es_params)
prob_state = problem.init(jr.PRNGKey(FIXED_PROBLEM_SEED))

# Trigger compilation
print("\n📊 Compiling run_evolution...")
(final_state, final_prob_state, final_rng), _ = run_evolution(state, prob_state, rng)
jax.block_until_ready(final_state.best_fitness)

# Get the lowered representation
print("\n🔬 Getting lowered HLO representation...")
lowered = jax.jit(run_evolution_impl).lower(state, prob_state, rng)

print("\n" + "=" * 80)
print("LOWERED HLO (first 5000 chars):")
print("=" * 80)
hlo_text = lowered.as_text()
print(hlo_text[:5000])
print("\n... (truncated)")
print(f"\nTotal HLO length: {len(hlo_text):,} characters")

# Save full HLO to file
hlo_filename = f"evosax_hlo_{PROBLEM_NAME}_{DIMENSIONS}D.txt"
with open(hlo_filename, 'w') as f:
    f.write(hlo_text)
print(f"\n💾 Full HLO saved to: {hlo_filename}")

print("\n" + "=" * 80)
print("✅ Evosax compilation analysis complete!")
print("=" * 80)

🔍 Analyzing Evosax Compilation...

📊 Compiling run_evolution...

📊 Compiling run_evolution...

🔬 Getting lowered HLO representation...

LOWERED HLO (first 5000 chars):
module @jit_run_evolution_impl attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<10xf32>, %arg1: tensor<f32>, %arg2: tensor<i32>, %arg3: tensor<100x10xf32>, %arg4: tensor<100xf32>, %arg5: tensor<f32>, %arg6: tensor<i32>, %arg7: tensor<2xui32>) -> (tensor<10xf32> {jax.result_info = "result[0][0].best_solution"}, tensor<f32> {jax.result_info = "result[0][0].best_fitness"}, tensor<i32> {jax.result_info = "result[0][0].generation_counter"}, tensor<100x10xf32> {jax.result_info = "result[0][0].population"}, tensor<100xf32> {jax.result_info = "result[0][0].fitness"}, tensor<f32> {jax.result_info = "result[0][0].std"}, tensor<i32> {jax.result_info = "result[0][1].counter"}, tensor<2xui32> {jax.result_info = "result[0][2]"}) {
    %c = stablehlo.constant dense<0> : te

In [33]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from typing import Dict, Any, Tuple

# ==========================================
# 1. CONFIGURATION (The "Quality" Map)
# ==========================================
CATEGORY_MAP = {
    # Heavy Compute
    'dot': 'Compute (Heavy)', 'convolution': 'Compute (Heavy)', 
    'threefry': 'Compute (RNG)', 'rng': 'Compute (RNG)', 'rand': 'Compute (RNG)',
    
    # ALU (Light Compute)
    'add': 'Compute (ALU)', 'subtract': 'Compute (ALU)', 'multiply': 'Compute (ALU)', 
    'divide': 'Compute (ALU)', 'power': 'Compute (ALU)', 'negate': 'Compute (ALU)',
    'sine': 'Compute (ALU)', 'cosine': 'Compute (ALU)', 'exp': 'Compute (ALU)', 
    'log': 'Compute (ALU)', 'rsqrt': 'Compute (ALU)', 'tanh': 'Compute (ALU)',
    'xor': 'Compute (ALU)', 'or': 'Compute (ALU)', 'and': 'Compute (ALU)', 
    'shift': 'Compute (ALU)', 'compare': 'Compute (ALU)', 'select': 'Compute (ALU)',
    'convert': 'Compute (ALU)', 'reduce': 'Compute (ALU)', 'map': 'Compute (ALU)',
    
    # Data Movement (Cheap)
    'broadcast': 'Data (Cheap)', 'reshape': 'Data (Cheap)', 'bitcast': 'Data (Cheap)',
    'slice': 'Data (Cheap)', 'pad': 'Data (Cheap)', 'concatenate': 'Data (Cheap)',
    'iota': 'Data (Cheap)', 'reverse': 'Data (Cheap)', 'transpose': 'Data (Cheap)',
    
    # Memory (Expensive)
    'copy': 'Memory (Expensive)', 'gather': 'Memory (Expensive)', 'scatter': 'Memory (Expensive)',
    
    # Control Flow
    'while': 'Control Flow', 'call': 'Control Flow', 'return': 'Control Flow', 
    'tuple': 'Control Flow', 'get-tuple-element': 'Control Flow', 'branch': 'Control Flow'
}

COLORS = {
    'Compute (Heavy)': '#d62728', # Red
    'Compute (RNG)': '#ff7f0e',   # Orange
    'Compute (ALU)': '#2ca02c',   # Green
    'Data (Cheap)': '#1f77b4',    # Blue
    'Memory (Expensive)': '#9467bd', # Purple
    'Control Flow': '#7f7f7f',    # Gray
    'Other': '#bcbd22'            # Yellow
}

# ==========================================
# 2. PARSING FUNCTION
# ==========================================
def parse_hlo(hlo_text: str) -> Dict[str, Any]:
    """
    Parses raw HLO text and returns a dictionary of stats.
    """
    # 1. Extract Operations using Regex
    # Matches "stablehlo.add", "mhlo.dot", "func.return", etc.
    op_pattern = r'\b(stablehlo|mhlo|chlo|func)\.([a-z0-9_]+)\b'
    matches = re.findall(op_pattern, hlo_text)
    
    # 2. Count Operations
    # We store just the operation name (e.g., 'add') for categorization, 
    # but keep the full name for the raw count.
    raw_counts = Counter([f"{ns}.{name}" for ns, name in matches])
    simple_counts = Counter([name for ns, name in matches])
    
    # 3. Categorize
    df_data = []
    for full_op, count in raw_counts.items():
        name = full_op.split('.')[1]
        # Fuzzy match for things like "shift_left" -> "shift"
        cat = 'Other'
        for key, val in CATEGORY_MAP.items():
            if key in name:
                cat = val
                break
        
        df_data.append({'Operation': full_op, 'Count': count, 'Category': cat})
    
    df = pd.DataFrame(df_data)
    if not df.empty:
        df = df.sort_values('Count', ascending=False)
    
    # 4. Calculate Density Score
    total_ops = df['Count'].sum() if not df.empty else 0
    compute_ops = df[df['Category'].str.contains('Compute')]['Count'].sum() if not df.empty else 0
    density = (compute_ops / total_ops * 100) if total_ops > 0 else 0
    
    return {
        'total_ops': total_ops,
        'compute_density': density,
        'counts': raw_counts,
        'dataframe': df
    }

# ==========================================
# 3. SINGLE ANALYSIS VISUALIZATION
# ==========================================
def visualize_single(hlo_stats: Dict[str, Any], title: str = "HLO Analysis"):
    """
    Visualizes a single HLO report.
    """
    df = hlo_stats['dataframe']
    if df.empty:
        print("No operations found in HLO text.")
        return

    print(f"\n📊 REPORT: {title}")
    print(f"Total Instructions: {hlo_stats['total_ops']}")
    print(f"Compute Density:    {hlo_stats['compute_density']:.2f}% (Higher is denser math)")
    print("-" * 40)
    print(df.head(10).to_string(index=False))
    
    # Plotting
    plt.figure(figsize=(12, 6))
    
    # 1. Category Breakdown (Pie)
    plt.subplot(1, 2, 1)
    cat_sums = df.groupby('Category')['Count'].sum()
    colors = [COLORS.get(x, '#333333') for x in cat_sums.index]
    plt.pie(cat_sums, labels=cat_sums.index, autopct='%1.1f%%', colors=colors, startangle=140)
    plt.title("Instruction Categories")
    
    # 2. Top 10 Ops (Bar)
    plt.subplot(1, 2, 2)
    top_10 = df.head(10)
    sns.barplot(data=top_10, x='Count', y='Operation', hue='Category', dodge=False, palette=COLORS)
    plt.title("Top 10 Heavy Hitters")
    plt.tight_layout()
    plt.show()

# ==========================================
# 4. COMPARISON VISUALIZATION
# ==========================================
def compare_hlo(stats_a: Dict[str, Any], stats_b: Dict[str, Any], names: Tuple[str, str] = ('Model A', 'Model B')):
    """
    Compares two HLO reports side-by-side.
    """
    df_a = stats_a['dataframe'].copy()
    df_b = stats_b['dataframe'].copy()
    
    df_a['Model'] = names[0]
    df_b['Model'] = names[1]
    
    # Merge for full comparison
    full_df = pd.merge(df_a, df_b, on=['Operation', 'Category'], how='outer', suffixes=('_A', '_B')).fillna(0)
    full_df['Diff'] = full_df['Count_A'] - full_df['Count_B']
    full_df['Total'] = full_df['Count_A'] + full_df['Count_B']
    
    # Sort by impact
    full_df = full_df.sort_values('Total', ascending=False).reset_index(drop=True)
    
    print(f"\n⚔️ COMPARISON: {names[0]} vs {names[1]}")
    print(f"{names[0]} Density: {stats_a['compute_density']:.2f}%")
    print(f"{names[1]} Density: {stats_b['compute_density']:.2f}%")
    print("-" * 60)
    print(full_df[['Operation', 'Category', 'Count_A', 'Count_B', 'Diff']].head(15).to_string(index=False))

    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Chart 1: Category Comparison
    cat_a = df_a.groupby('Category')['Count'].sum().reset_index()
    cat_b = df_b.groupby('Category')['Count'].sum().reset_index()
    cat_a['Model'] = names[0]
    cat_b['Model'] = names[1]
    cat_comp = pd.concat([cat_a, cat_b])
    
    sns.barplot(data=cat_comp, x='Category', y='Count', hue='Model', ax=axes[0])
    axes[0].set_title("Instruction Count by Category")
    axes[0].tick_params(axis='x', rotation=45)
    
    # Chart 2: Top Op Differences
    # Filter for ops with significant differences (>5)
    diff_plot = full_df[abs(full_df['Diff']) > 0].head(10)
    
    sns.barplot(data=diff_plot, y='Operation', x='Diff', hue='Category', dodge=False, palette=COLORS, ax=axes[1])
    axes[1].set_title(f"Difference ({names[0]} - {names[1]})")
    axes[1].set_xlabel(f"Positive = {names[0]} has more")
    
    plt.tight_layout()
    plt.show()